In [7]:
# Universidade Presbiteriana Mackenzie - Faculdade de Computacao e Informatica
# Disciplina: Inteligência Artificial - 7J SI - Prof. Dr. Leandro Zerbinatti
#
# Projeto: Análise de Padrões de Escolha e Desempenho de Heróis em Dota 2 por Faixa de Habilidade (MMR)
#
# Integrantes do grupo:
#   - David Haim Raiber | RA 10395618 | 10395618@mackenzista.com.br
#   - Isadora Caetano Brandão de Sousa    | RA 10420646 | 10420646@mackenzista.com.br
#   - Jennifer Aparecida de Sousa Tondade | RA 10420574 | 10420574@mackenzista.com.br
#   - Lucas Lacerda Gomes | RA 10322644 | 10322644@mackenzista.com.br
#
# Síntese do conteúdo deste arquivo:
#   Notebook de preparação dos dados coletados via OpenDota API que contém a
#   transformação da coluna de papéis em variáveis binárias (multi-hot
#   encoding), cálculo do percentual de picks normalizado por faixa de MMR
#   (pick rate) e engenharia das variáveis de desempenho médio e de variação
#   de desempenho entre as faixas disponíveis.
#
# Histórico de atualizações:
#
#   Data       | Autor                                  | Descrição
#   -----------|----------------------------------------|----------------------------------------
#   06/09/2026 | David, Isadora, Jennifer e Lucas       | Criação do notebook: Encoding de Roles,
#              |                                        | calcúlo de pick rate normalizado e
#              |                                        | engenharia de features de desempenho
#   13/09/2026 | David, Isadora, Jennifer e Lucas       | Verificação da funcionalidade do código.

# Notebook 2: Tratamento dos Dados de Heróis do Dota 2 (OpenDota API)

Este notebook realiza a preparação dos dados coletados via dota2_herostats_opendota_live.csv, com base nas observações do notebook de análise exploratória visualizacao_dos_dados_dota2.ipynb: O dataset possui apenas uma coluna com valores ausentes e a coluna roles precisa ser transformada e novas variáveis precisam ser derivadas para uso na etapa de modelagem (N2). Como a faixa **Immortal** não possui dados 0 picks e 0 vitórias em todos os heróis, limitação do endpoint no momento da coleta, ela é excluída dos cálculos de desempenho abaixo.

In [8]:
import pandas as pd
import numpy as np

BRACKETS = ["Herald", "Guardian", "Crusader", "Archon", "Legend", "Ancient", "Divine", "Immortal"]
# Immortal excluída dos cálculos de desempenho
BRACKETS_VALIDAS = [b for b in BRACKETS if b != "Immortal"]
ATTR_MAP = {"str": "Forca", "agi": "Agilidade", "int": "Inteligencia", "all": "Universal"}

df = pd.read_csv("dota2_herostats_opendota_live.csv")
print(f"Dimensões do dataset: {df.shape}")
df.head()

Dimensões do dataset: (92, 28)


,hero_id,hero_name,primary_attr,roles,picks_Herald,wins_Herald,winrate_Herald,picks_Guardian,wins_Guardian,winrate_Guardian,...,winrate_Legend,picks_Ancient,wins_Ancient,winrate_Ancient,picks_Divine,wins_Divine,winrate_Divine,picks_Immortal,wins_Immortal,winrate_Immortal
0,1,Anti-Mage,agi,"Carry,Escape,Nuker",15444,7560,48.95,50643,25221,49.80,...,49.43,56831,28132,49.50,29214,14438,49.42,0,0,NaN
1,2,Axe,str,"Initiator,Durable,Disabler,Carry",19924,10462,52.51,70576,36362,51.52,...,49.96,118005,58437,49.52,73308,35562,48.51,0,0,NaN
2,3,Bane,all,"Support,Disabler,Nuker,Durable",4361,2081,47.72,14181,6813,48.04,...,50.16,25082,12914,51.49,22046,11370,51.57,0,0,NaN
3,4,Bloodseeker,agi,"Carry,Disabler,Nuker,Initiator",10432,5362,51.40,24136,12309,51.00,...,51.82,14418,7345,50.94,7218,3740,51.81,0,0,NaN
4,5,Crystal Maiden,int,"Support,Disabler,Nuker",19299,10496,54.39,63841,34021,53.29,...,51.25,82336,41702,50.65,49943,24937,49.93,0,0,NaN


In [9]:
# Multi-hot encoding da coluna roles
roles_dummies = df["roles"].str.get_dummies(sep=",")
roles_dummies.columns = [f"Role_{c}" for c in roles_dummies.columns]
roles_dummies.head()

,Role_Carry,Role_Disabler,Role_Durable,Role_Escape,Role_Initiator,Role_Nuker,Role_Pusher,Role_Support
0,1,0,0,1,0,1,0,0
1,1,1,1,0,1,0,0,0
2,0,1,1,0,0,1,0,1
3,1,1,0,0,1,1,0,0
4,0,1,0,0,0,1,0,1


In [10]:
# Percentual de picks normalizado por faixa de MMR (pick rate dentro do bracket)
pickrate_cols = {}
for b in BRACKETS_VALIDAS:
    total_bracket = df[f"picks_{b}"].sum()
    pickrate_cols[f"PickRate_{b}"] = 100 * df[f"picks_{b}"] / total_bracket
pickrate_df = pd.DataFrame(pickrate_cols)
pickrate_df.head()

,PickRate_Herald,PickRate_Guardian,PickRate_Crusader,PickRate_Archon,PickRate_Legend,PickRate_Ancient,PickRate_Divine
0,1.604780,1.589922,1.559947,1.455760,1.273827,1.098553,0.840209
1,2.070295,2.215712,2.301927,2.339222,2.339328,2.281056,2.108373
2,0.453150,0.445208,0.444914,0.448961,0.451392,0.484839,0.634053
3,1.083985,0.757742,0.557415,0.429177,0.344238,0.278702,0.207593
4,2.005351,2.004269,1.873770,1.734788,1.649194,1.591569,1.436384


In [11]:
# Engenharia de variáveis de desempenho
winrate_cols = [f"winrate_{b}" for b in BRACKETS_VALIDAS]
winrate_mean = df[winrate_cols].mean(axis=1)
winrate_std = df[winrate_cols].std(axis=1)
# Delta: Diferença entre a faixa mais alta disponível (Divine) e a mais baixa (Herald),
winrate_delta = df["winrate_Divine"] - df["winrate_Herald"]
pickrate_mean = pickrate_df.mean(axis=1)

df_tratado = pd.DataFrame({
    "Name": df["hero_name"],
    "Primary Attribute": df["primary_attr"].map(ATTR_MAP),
    "WinRate_Mean": winrate_mean.round(4),
    "WinRate_Std": winrate_std.round(4),
    "WinRate_Delta": winrate_delta.round(2),
    "PickRate_Mean": pickrate_mean.round(4),
})
df_tratado = pd.concat([df_tratado, roles_dummies], axis=1)
df_tratado

,Name,Primary Attribute,WinRate_Mean,WinRate_Std,WinRate_Delta,PickRate_Mean,Role_Carry,Role_Disabler,Role_Durable,Role_Escape,Role_Initiator,Role_Nuker,Role_Pusher,Role_Support
0,Anti-Mage,Agilidade,49.6114,0.4212,0.47,1.3461,1,0,0,1,0,1,0,0
1,Axe,Forca,50.5214,1.3346,-4.00,2.2366,1,1,1,0,1,0,0,0
2,Bane,Universal,49.5629,1.5566,3.85,0.4804,0,1,1,0,0,1,0,1
3,Bloodseeker,Agilidade,51.3157,0.3758,0.41,0.5227,1,1,0,0,1,1,0,0
4,Crystal Maiden,Inteligencia,51.8871,1.5569,-4.46,1.7565,0,1,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,Naga Siren,Agilidade,48.7214,0.9829,-0.92,0.2215,1,1,0,1,1,0,1,1
88,Keeper of the Light,Inteligencia,49.8171,1.9653,5.39,0.8343,0,1,0,0,0,1,0,1
89,Io,Universal,48.7714,0.5005,1.05,0.5619,0,0,0,1,0,1,0,1
90,Visage,Universal,52.3071,2.6284,7.02,0.1888,0,1,1,0,0,1,1,1


In [12]:
# Exportar base de dados tratada, para reaproveitamento na etapa de modelagem (N2)
df_tratado.to_csv("dota2_hero_preference_tratado.csv", index=False, encoding="utf-8")
print("Dataset tratado salvo: dota2_hero_preference_tratado.csv")
print(f"Linhas: {len(df_tratado)} | Colunas: {list(df_tratado.columns)}")

Dataset tratado salvo: dota2_hero_preference_tratado.csv
Linhas: 92 | Colunas: ['Name', 'Primary Attribute', 'WinRate_Mean', 'WinRate_Std', 'WinRate_Delta', 'PickRate_Mean', 'Role_Carry', 'Role_Disabler', 'Role_Durable', 'Role_Escape', 'Role_Initiator', 'Role_Nuker', 'Role_Pusher', 'Role_Support']
